In [2]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from functions import *
import sys, os

## Sundstrom et al. 2025
https://arxiv.org/pdf/2505.04577 

https://github.com/msundstrom33/ComparingActiveLearningMethods

In [3]:
### Taking the average of COPUS COPUS_codes_sundstrom_weir across multiple observations of the same class

path = 'Published Data/Sundstrom_2025_data/data/classroom-observations/'

COPUS_sundstrom = pd.DataFrame()
unique_courses = []

#Get course name
for file in os.listdir(path):
    name = file.split('.')[0]
    name = file.split('_')
    unique_courses.append(name[0] + '_' + name[1])

unique_courses = list(set(unique_courses))
#if starts with 'LPA' remove, these are analysis files
unique_courses = [course for course in unique_courses if not course.startswith('LPA')]

for course in unique_courses:
    course_files = [f for f in os.listdir(path) if f.startswith(course)]
    course_data = pd.DataFrame()
    
    #stack all observations for a given class into one dataframe
    for file in course_files:
        data = pd.read_csv(os.path.join(path, file), index_col=0, header=8)
        #drop unnamed columns
        data = data.loc[:, ~data.columns.str.contains('^Unnamed')]
        #drop last 5 columns
        data = data.iloc[:, :-5]
        
        #drop rows where L is L ( get rid of non-numeric rows)
        data = data[data['L'] != 'L']
          
        #remove all nan columns and rows
        data = data.dropna(axis=0, how='all')
        course_data = pd.concat([course_data, data], ignore_index=True)
    #replace nans with 0
    course_data = course_data.infer_objects(copy = False).fillna(0)
    #make all values numeric
    course_data = course_data.apply(pd.to_numeric, errors='coerce')

    #find total length of observations
    Tot = len(course_data)
    #average all and add to COPUS_sundstrom
    COPUS_sundstrom = pd.concat([COPUS_sundstrom, course_data.mean().to_frame().T], ignore_index=True)
    #add total number of observations to COPUS_sundstrom
    COPUS_sundstrom.loc[len(COPUS_sundstrom)-1, 'Total'] = Tot
    #make first column the course_id
    COPUS_sundstrom.loc[len(COPUS_sundstrom)-1, 'Course'] = course

#make 'Course' the first column
cols = COPUS_sundstrom.columns.tolist()
cols = ['Course'] + [col for col in cols if col != 'Course']
COPUS_sundstrom = COPUS_sundstrom[cols]
#sort by course
COPUS_sundstrom = COPUS_sundstrom.sort_values('Course').reset_index(drop=True)

In [14]:
## Loading in concept inventory data
path = 'Published Data/Sundstrom_2025_data/data/rosters/'

CI_statistics_sundstrom = pd.DataFrame()
CI_statistics_sundstrom['Course'] = COPUS_sundstrom['Course']

max_scores = np.array([30, 30, 30, 30, 47, 0,##ISLE
 47, 47, 26, 26, 14, 30, 50, 47, 30, ##PI
 0, 0, 30, 30, 30, 30, 25, ##scale up
 26, 47, 33, 30, 26, 30, 30, 0]) ## Tutorials

courses = CI_statistics_sundstrom['Course'].values
for i in range(len(courses)):
    course = courses[i]

    #read in scores for the course
    file = path + 'Roster_' + course + '.csv'

    try:  #some COPUS courses do not have CI data
        CI_scores = pd.read_csv(path + 'Roster_' + course + '.csv')
        #get the class size
        N = int(len(CI_scores))
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'N'] = N

        #We are only using matched scores, so we drop any rows with missing pre or post scores
        CI_scores.dropna(inplace = True)
 

        pretest_scores = CI_scores['PreScore'].values/max_scores[i]
        posttest_scores = CI_scores['PostScore'].values/max_scores[i]

        #add means and STDs to CI_statistics_sundstrom
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PreMean'] = np.mean(pretest_scores)
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PreStd'] = np.std(pretest_scores)
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PostMean'] = np.mean(posttest_scores)
        CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'] == course, 'PostStd'] = np.std(posttest_scores)
    except:
        #If there is no CI data for the course, we remove it from the CI_statistics_sundstrom dataframe
        CI_statistics_sundstrom = CI_statistics_sundstrom[CI_statistics_sundstrom['Course'] != course]

#calculate effect sizes and add to CI_statistics_sundstrom
CI_statistics_sundstrom['Effect Size'] = [effect_size(CI_statistics_sundstrom['PreMean'].iloc[i],
                                         CI_statistics_sundstrom['PostMean'].iloc[i],
                                         CI_statistics_sundstrom['PreStd'].iloc[i],
                                         CI_statistics_sundstrom['PostStd'].iloc[i]) for i in range(len(CI_statistics_sundstrom))]
#mark all courses as physics by default, then change to astronomy for the two astronomy courses
CI_statistics_sundstrom['Field'] = 'Physics'
astro_courses = ['Tutorials_Course2', 'Tutorials_Course6',]
CI_statistics_sundstrom.loc[CI_statistics_sundstrom['Course'].isin(astro_courses), 'Field'] = 'Astronomy'

#mark source
CI_statistics_sundstrom['Source'] = 'Sundstrom et al. 2025'

sundstrom_combined = pd.merge(COPUS_sundstrom, CI_statistics_sundstrom, on='Course')
#save to csv
sundstrom_combined.to_csv('Published Data/Sundstrom_2025_data/combined_sundstrom_data.csv', index=False)

## Weir et al. 2019

https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0220900

In [15]:
path = 'Published Data/Weir_2019_data/'

#read in COPUS codes
file = pd.ExcelFile(path + 'UBC_raw_COPUS.xlsx')
COPUS_codes_weir = pd.read_excel(file, sheet_name=file.sheet_names[-1])

#select only relevant columns, rename them, and convert values to percentages
COPUS_codes_weir = COPUS_codes_weir[['unique.section','num.students', 'ICQ', 'IPQ','SL', 'Ilec', 'SCG', 'SWG', 'SOG', 'SQ', 'I1o1' ]]
COPUS_codes_weir.columns = ['Course', 'N', 'CQ', 'PQ','L', 'Lec', 'CG', 'WG', 'OG', 'SQ', '1o1']

COPUS_codes_weir.iloc[:, 2:] = COPUS_codes_weir.iloc[:, 2:].div(100)
COPUS_codes_weir['Field'] = 'Biology'


In [16]:
# load in CI statistics


CI_statistics_weir = pd.DataFrame()
#load raw scores, keep only relevant columns, convert to percentages
raw_scores = pd.read_excel(path + 'UBC_raw_scores.xlsx')
raw_scores.drop(columns=['course.level', 'term'], inplace=True)
raw_scores.iloc[:, 1:] = raw_scores.iloc[:, 1:].div(100)

course_list = raw_scores['unique.section'].unique()
for course in course_list:
    course_scores = raw_scores[raw_scores['unique.section'] == course]
    pretest_scores = course_scores['pre.score'].values
    posttest_scores = course_scores['post.score'].values
    CI_statistics_weir.loc[CI_statistics_weir.shape[0], 'Course'] = course
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PreMean'] = np.mean(pretest_scores)
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PreStd'] = np.std(pretest_scores)
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PostMean'] = np.mean(posttest_scores)
    CI_statistics_weir.loc[CI_statistics_weir.shape[0]-1, 'PostStd'] = np.std(posttest_scores)

CI_statistics_weir['Effect Size'] = [effect_size(CI_statistics_weir['PreMean'].iloc[i],
                                         CI_statistics_weir['PostMean'].iloc[i],
                                         CI_statistics_weir['PreStd'].iloc[i],
                                         CI_statistics_weir['PostStd'].iloc[i])
                                     for i in range(len(CI_statistics_weir))]

CI_statistics_weir['Source'] = 'Weir et al. 2019'

In [17]:
## combine COPUS codes with CI statistics
weir_combined = pd.merge(COPUS_codes_weir, CI_statistics_weir, on = 'Course')

## Connell et al 2017

https://www.lifescied.org/doi/full/10.1187/cbe.15-03-0062

In [18]:
#These values come from using webplot digitizer on from the graph  of COPUS codes in Connell et al. 2017
connell_combined = pd.DataFrame({
    'Course': ['Connell_ext', 'Connell_mod'],
    'L': [.65, .87],
    'Lec': [0.34, 0.75], #
    'PQ': [0.26, 0.28],#
    'MG': [0.31, 0.0], #
    'CQ': [0.0, 0.0], #not reported
    '1o1': [.318, .004],
    
    
    'CG': [0, 0], #not reported
    'WG': [0.4, 0.0],
    'OG': [0.12, 0.66], 
    'SQ': [0.12, 0.024], 
    
    'PreMean': [0.443, 0.429],
    'PostMean': [0.585, 0.546],
    'PreStd': [0.107, 0.104],
    'PostStd': [0.121, 0.122],

    'N': [182, 172], #reported 316 took survey
    'Field': ['Biology', 'Biology'],
})

connell_combined['Effect Size'] = (connell_combined['PostMean'] - connell_combined['PreMean']) / np.sqrt(
    (connell_combined['PreStd']**2 + connell_combined['PostStd']**2) / 2)

connell_combined['Source'] = 'Connell et al. 2017'
connell_combined

,Course,L,Lec,PQ,MG,CQ,1o1,CG,WG,OG,SQ,PreMean,PostMean,PreStd,PostStd,N,Field,Effect Size,Source
0,Connell_ext,0.65,0.34,0.26,0.31,0.0,0.318,0,0.4,0.12,0.120,0.443,0.585,0.107,0.121,182,Biology,1.243272,Connell et al. 2017
1,Connell_mod,0.87,0.75,0.28,0.00,0.0,0.004,0,0.0,0.66,0.024,0.429,0.546,0.104,0.122,172,Biology,1.032130,Connell et al. 2017


### New data

In [20]:
measured_data = pd.read_csv('Measured Data/measured_data.csv')
measured_data['Source'] = 'Measured Data'

## Combining Alll Datasets

In [21]:
all_data = pd.concat([sundstrom_combined, weir_combined, connell_combined, measured_data], ignore_index=True)
#reorder columns
#We keep these 8 COPUS codes because they were found by Stains et al (2018) to be the most informative for characterizing STEM classrooms
all_data = all_data[['Source', 'Course', 'N', 'Field', 'PreMean', 'PostMean', 'PreStd', 'PostStd', 'Effect Size',
                                         'Lec', 'CG', 'CQ', 'PQ', 'WG', 'OG', 'SQ', '1o1' ]]

#When the mean of pretest scores is above 60, it imposes a ceiling effect on effect sizes
# and indicates that the CI may not be appropriate for that course, so we remove those courses
all_data = all_data[all_data['PreMean'] <= 0.6]

#saved combined data to csv
all_data.to_csv('combined_COPUS_CI_data.csv', index=False)
all_data

,Source,Course,N,Field,PreMean,PostMean,PreStd,PostStd,Effect Size,Lec,CG,CQ,PQ,WG,OG,SQ,1o1
0,Sundstrom et al. 2025,ISLE_Course1,150.0,Physics,0.278744,0.441546,0.177654,0.195802,0.870839,0.398230,0.159292,0.176991,0.814159,0.000000,0.150442,0.203540,0.088496
1,Sundstrom et al. 2025,ISLE_Course2,16.0,Physics,0.413889,0.461111,0.195533,0.207201,0.234410,0.628099,0.000000,0.057851,0.223140,0.107438,0.033058,0.148760,0.000000
2,Sundstrom et al. 2025,ISLE_Course3,118.0,Physics,0.345318,0.423596,0.205252,0.175190,0.410227,0.533898,0.127119,0.254237,0.703390,0.000000,0.101695,0.262712,0.000000
3,Sundstrom et al. 2025,ISLE_Course4,15.0,Physics,0.423810,0.645238,0.232457,0.213663,0.991806,0.123894,0.000000,0.000000,0.920354,0.424779,0.106195,0.283186,0.044248
4,Sundstrom et al. 2025,ISLE_Course5,32.0,Physics,0.248227,0.476359,0.090269,0.290706,1.059888,0.165605,0.000000,0.000000,0.859873,0.350318,0.242038,0.006369,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66,Measured Data,4b,175.0,Physics,0.378000,0.556000,0.154000,0.246000,0.867354,0.377000,0.506000,0.558000,0.286000,0.000000,0.000000,0.143000,NaN
67,Measured Data,4c,175.0,Physics,0.319000,0.466000,0.163000,0.235000,0.726895,0.368000,0.461000,0.579000,0.224000,0.000000,0.000000,0.171000,NaN
68,Measured Data,4d,175.0,Physics,0.370000,0.543000,0.160000,0.261000,0.799176,0.551000,0.551000,0.538000,0.244000,0.000000,0.000000,0.051000,NaN
69,Measured Data,4e,175.0,Physics,0.328000,0.503000,0.154000,0.242000,0.862792,0.500000,0.462000,0.500000,0.321000,0.000000,0.000000,0.141000,NaN
